In [5]:
import pandas as pd
import scipy as sp
import numpy as np
from lightfm import cross_validation
from lightfm.evaluation import precision_at_k
from lightfm.evaluation import recall_at_k
from lightfm.evaluation import auc_score
from lightfm import LightFM
import matplotlib.pyplot as plt
import pickle
import collections
import seaborn as sns
import gzip
import textwrap
import os
from datetime import date
import time
import sys

**Create trait vs SNP matrix**

In [2]:
#load data
data = []
with open("/home/nasi4/proj/GWAS/UKBB_201807/results/all.UKBB.hg38.sig.phenotype.bed")as f:
    for line in f:
        data.append(line.strip().split())

#combine label elements
for row in data:
    row.append(' '.join(row[5:]))
    del row[5:-1]
    
#create dataframe
df = pd.DataFrame(data, columns = ['Chromosome', 'p1', 'p2', 'Reference SNP cluster ID', 'p-value', 'Trait'])
d = df
# df = df[df['p-value']!='0.00000e+00']
# test_list = df['p-value'].to_list()
# test_list = list(map(float, test_list))
# print(max(test_list))

# #calculating the negative log of the p-value, setting infinite values to the max non-infinite value observed, and
# #normalizing by max -log(p-value) score
# pval_mlog = -np.log(list(map(float, list(df['p-value']))))

# inf_mask = np.ma.array(pval_mlog, mask=~np.isfinite(pval_mlog))
# max_ind = np.ma.argmax(inf_mask, fill_value=-np.inf)
# print(max_ind)
# max_val = pval_mlog[max_ind]
# print(max_val)

# for ind in range(len(pval_mlog)):
#     if pval_mlog[ind] > pval_mlog[max_ind]:
#         pval_mlog[ind] = pval_mlog[max_ind]
        
# df['pval_mlog'] = pval_mlog/pval_mlog[max_ind]

# df[(df['Trait']=='Weight')&(df['Reference SNP cluster ID']=='16:53812770_ATTTT_A')]

# # Reference SNP cluster ID: rs11642015, chr16:53768582-53768583, p-value = 1.14192e-191

# #creating df with SNP, trait, and p-value
# ts_df = df[['Reference SNP cluster ID', 'Trait', 'pval_mlog']]

# #finding unique traits and SNPs
# SNP_values = np.unique(ts_df['Reference SNP cluster ID'].values)
# trait_values = np.unique(ts_df['Trait'].values)

# ts = ts_df.groupby(['Reference SNP cluster ID', 'Trait']).max()

# #20211129 - changed from original way I did this. 
# # I was averaging all -ln(p-values) from a given trait vs SNP combination
# # now I am taking the max valeu
# # ts = ts_df.groupby(['Reference SNP cluster ID', 'Trait']).mean()
# ts = ts.reset_index()

# ts['trait_ind'] = ts.groupby(['Trait']).ngroup()
# ts['snp_ind'] = ts.groupby(['Reference SNP cluster ID']).ngroup()

# columns = ts['snp_ind'].to_list()
# rows = ts['trait_ind'].to_list()
# pval_mlog_data = ts['pval_mlog'].to_list()

# data_coo = sp.sparse.coo_matrix((pval_mlog_data, (rows, columns)), shape=(len(trait_values), len(SNP_values)))

5e-08
736764
743.7469247408213


/tmp/ipykernel_17583/461438952.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['pval_mlog'] = pval_mlog/pval_mlog[max_ind]


In [ ]:
len(SNP_values)

In [3]:
data_matrix = data_coo.toarray()

In [6]:
sys.getsizeof(data_matrix)

63793819720

In [8]:
data_matrix.shape

(3815, 2090230)

In [8]:
pickle_out = open('/home/ctrader/Hold/GWAS/GWAS_trait_values_CT20220110.pickle', 'wb')
pickle.dump(trait_values, pickle_out)
pickle_out.close()

In [2]:
pickle_in = open('/home/ctrader/Hold/GWAS/GWAS_trait_values_CT20220110.pickle', 'rb')
trait_values=pickle.load(pickle_in)

In [3]:
trait_values

array(['#Arthrosis', '#Deforming dorsopathies', '#Other joint disorders',
       ..., 'malignant neoplasm of female genital organs',
       'malignant neoplasm of male genital organs',
       'severe traumatic brain injury, does not include concussion'],
      dtype=object)

**Load Model**

In [42]:
pickle_in = open('/home/ctrader/Hold/GWAS/gwas_model_e30_d10_reg00001_CT20211006.pickle', 'rb')
gwas_model_e30_d10_reg00001=pickle.load(pickle_in)

pickle_in = open('/home/ctrader/Hold/GWAS/gwas_data_e30_d10_reg00001_CT20211006.pickle', 'rb')
data_e30_d10_reg00001=pickle.load(pickle_in)

In [4]:
# snp_b, snp_e = gwas_model_e30_d10_reg00001.get_item_representations()
# snp_e = snp_e.T
# trait_b, trait_e = gwas_model_e30_d10_reg00001.get_user_representations()

In [5]:
# n_dimensions, n_snps = snp_e.shape
# n_traits, n_dimensions = trait_e.shape

In [6]:
# gwas_model_e30_d10_reg00001.predict(1600, np.arange(n_snps))

In [7]:
# predictions = ((np.dot(trait_e, snp_e) + snp_b).T + trait_b).T

In [8]:
# predictions_sorted = np.argsort(-predictions, axis=1)

In [11]:
# predictions_sorted_100 = predictions_sorted[:, :100]

In [60]:
predictions_sorted_100 = generate_100_recs(gwas_model_e30_d10_reg00001)

NameError: name 'generate_100_recs' is not defined

In [77]:
pickle_out = open('/home/ctrader/Hold/GWAS/pickled_subthreshold_data/top_100_SNP_recommendations_for_each_trait_all_traits_CT20220112.pickle', 'wb')
pickle.dump(predictions_sorted_100, pickle_out)
pickle_out.close()

In [61]:
pickle_in = open('/home/ctrader/Hold/GWAS/pickled_subthreshold_data/top_100_SNP_recommendations_for_each_trait_all_traits_CT20220112.pickle', 'rb')
predictions_sorted_100=pickle.load(pickle_in)

In [5]:
ts_trait_based = ts.groupby(['Trait']).first()
trait_values_order_to_match_subthreshold = list(ts_trait_based.loc[np.array(trait_values)]['trait_ind'])
ts_snp_ind_based = ts.groupby(['snp_ind']).first()
df_snp_id_based = df.groupby(['Reference SNP cluster ID']).first()

In [123]:
#before p-value was found for each recommended SNP
top_100_recommended_snps_all_traits_chrm_list = []
top_100_recommended_snps_all_traits_start_list = []
top_100_recommended_snps_all_traits_end_list = []
top_100_recommended_snps_all_traits_order_list = []
top_100_recommended_snps_all_traits_id_list = []
j=0
for i in trait_values_order_to_match_subthreshold:
    #getting information for all SNP recs for each trait
    snp_info_df = df_snp_id_based.loc[ts_snp_ind_based.loc[predictions_sorted_100[i]]['Reference SNP cluster ID']]
    top_100_recommended_snps_all_traits_chrm_list.extend(list(snp_info_df['Chromosome']))
    top_100_recommended_snps_all_traits_start_list.extend(list(snp_info_df['p1']))
    top_100_recommended_snps_all_traits_end_list.extend(list(snp_info_df['p2']))
    top_100_recommended_snps_all_traits_order_list.extend(list(100*j + np.arange(100)))
    top_100_recommended_snps_all_traits_id_list.extend(list(snp_info_df.index))
    j+=1

In [124]:
pickle_out = open('/home/ctrader/Hold/GWAS/pickled_subthreshold_data/top_100_SNP_recommendations_for_each_trait_all_traits_id_list_CT20220124.pickle', 'wb')
pickle.dump(top_100_recommended_snps_all_traits_id_list, pickle_out)
pickle_out.close()

In [70]:
pickle_in = open('/home/ctrader/Hold/GWAS/pickled_subthreshold_data/top_100_SNP_recommendations_for_each_trait_all_traits_id_list_CT20220124.pickle', 'rb')
top_100_recommended_snps_all_traits_id_list=pickle.load(pickle_in)

In [15]:
# top_100_recommended_snps_all_traits_pval_list = []
# for i in trait_values_order_to_match_subthreshold:
#     snp_info_df = df_snp_id_based.loc[ts_snp_ind_based.loc[predictions_sorted_100[i]]['Reference SNP cluster ID']]

In [102]:
tv = trait_values[0:5]

In [24]:
print(trait_df[trait_df['Reference SNP cluster ID']==rec_snp_id]['p-value'])

4512595    0.000004
Name: p-value, dtype: float64


In [26]:
trait_df

NameError: name 'trait_df' is not defined

In [77]:
i=0
rec_snp_id = top_100_recommended_snps_all_traits_id_list[(i*100):(i*100)+100]

trait_df_snp_ind_based = trait_df.set_index('Reference SNP cluster ID')
trait_df_snp_ind_based = trait_df_snp_ind_based.loc[trait_df_snp_ind_based.index.isin(rec_snp_id)]
trait_df_snp_ind_based = trait_df_snp_ind_based.sort_values(['p-value'])
trait_df_snp_ind_based = trait_df_snp_ind_based.groupby(['Reference SNP cluster ID']).first()

trait_df_snp_ind_based = trait_df_snp_ind_based.loc[rec_snp_id]
pvals_for_rec_snps.extend(trait_df_snp_ind_based['p-value'])

In [148]:
trait_num = 0




tv = trait_values[:5]




#load traits and phenotypes
trait_codes = []
with gzip.open('/home/nasi4/proj/GWAS/UKBB_201807/info/phenotypes.both_sexes.tsv.gz','r') as f:
    for line in f:
        trait_codes.append(line.decode("utf-8").strip('\n').split('\t'))

trait_codes_df = pd.DataFrame(trait_codes[1:], columns=trait_codes[0])

trait_codes_df['description'] = trait_codes_df['description'].str.replace(')', ' ')
trait_codes_df['description'] = trait_codes_df['description'].str.replace('(', ' ')
trait_codes_df['description'] = trait_codes_df['description'].str.replace('-', ' ')
trait_codes_df['description'] = trait_codes_df['description'].str.replace('[', ' ')
trait_codes_df['description'] = trait_codes_df['description'].str.replace(']', ' ')
trait_codes_df['description'] = trait_codes_df['description'].str.replace('+', ' ')
trait_codes_df['description'] = trait_codes_df['description'].str.replace('_', ' ')
trait_codes_df['description'] = trait_codes_df['description'].str.replace(' ', '')

#find the phenotype for each trait
phenotype_list = []
not_added = []
for original_trait in tv:
    trait = original_trait.replace(')', ' ').replace('(', ' ').replace('-', ' ').replace('[', ' ').replace(']', ' ').replace('+', ' ').replace('_', ' ').replace(' ', '')
    if len(trait_codes_df[trait_codes_df['description']==(trait)])==1:
        phenotype_list.append(list(trait_codes_df[trait_codes_df['description']==(trait)]['phenotype'])[0])
    elif ((len(trait_codes_df[trait_codes_df['description']==(trait)])==2) & (len(trait_codes_df[(trait_codes_df['description']==(trait)) & (trait_codes_df['phenotype'].str.contains('irnt'))])==1)):
        phenotype_list.append(list(trait_codes_df[(trait_codes_df['description']==(trait)) & (trait_codes_df['phenotype'].str.contains('irnt'))]['phenotype'])[0])
    else:
        phenotype_list.append('SKIP_CT20220107')
        not_added.append(original_trait)

#search through full data to find the p-values for each recommended SNP for each trait
pvals_for_rec_snps = []
snp_id_check = []
for phenotype, trait, i in zip(phenotype_list, tv, range(len(phenotype_list))):
    i = i + trait_num
    if phenotype != 'SKIP_CT20220107':
        t = time.time()
        print(trait, phenotype)
        trait_data = []
        with gzip.open('/home/nasi4/proj/GWAS/UKBB_201807/phenotypes/' + phenotype + '.gwas.imputed_v3.both_sexes/' + phenotype + '.hg38.bed.gz','r') as f:
            for line in f:
                trait_data.append(line.decode("utf-8").strip('\n').split('\t'))
        trait_df = pd.DataFrame(trait_data, columns=['Chromosome', 'Start', 'End', 'Reference SNP cluster ID', 'p-value'])
        trait_df['p-value'] = trait_df['p-value'].astype('float')

        rec_snp_ids_one_trait = top_100_recommended_snps_all_traits_id_list[(i*100):(i*100)+100]
        
        trait_df_snp_ind_based = trait_df.set_index('Reference SNP cluster ID')
        trait_df_snp_ind_based = trait_df_snp_ind_based.loc[trait_df_snp_ind_based.index.isin(rec_snp_ids_one_trait)]
        trait_df_snp_ind_based = trait_df_snp_ind_based.sort_values(['p-value'])
        trait_df_snp_ind_based = trait_df_snp_ind_based.groupby(['Reference SNP cluster ID']).first()
        
        if len(trait_df_snp_ind_based)==100:
            trait_df_snp_ind_based = trait_df_snp_ind_based.loc[rec_snp_ids_one_trait]
            pvals_for_rec_snps.extend(list(trait_df_snp_ind_based['p-value']))
            snp_id_check.extend(list(trait_df_snp_ind_based.index))
        
        else:
            trait_df_snp_ind_based['SNP ID Col 2'] = trait_df_snp_ind_based.index
            for snp_id in rec_snp_ids_one_trait:
                snp_df = trait_df_snp_ind_based[trait_df_snp_ind_based['SNP ID Col 2']==snp_id]
                snp_id_check.append(snp_id)
                if len(snp_df)==1:
                    pvals_for_rec_snps.append(snp_df['p-value'])
                else:
                    pvals_for_rec_snps.append(np.nan)
                    
        print(time.time()-t)
    else:
        pvals_for_rec_snps.extend(list(np.array(['SKIP_CT20220107']*100)))
        snp_id_check.extend(list(np.array(['SKIP_CT20220107']*100)))

#pickle out the p-values and traits 
# pickle_out = open('/home/ctrader/Hold/GWAS/pickled_subthreshold_data/recommended_SNPs_for_each_trait_'+str(trait_num)+'_through_'+str(trait_num+4)+'_CT20220124.pickle', 'wb')
# pickle.dump(snp_id_check, pickle_out)
# pickle_out.close()

# pickle_out = open('/home/ctrader/Hold/GWAS/pickled_subthreshold_data/recommended_SNP_pvals_for_each_trait_'+str(trait_num)+'_through_'+str(trait_num+4)+'_CT20220124.pickle', 'wb')
# pickle.dump(pvals_for_rec_snps, pickle_out)
# pickle_out.close()

/tmp/ipykernel_138476/480664581.py:19: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  trait_codes_df['description'] = trait_codes_df['description'].str.replace(')', ' ')
/tmp/ipykernel_138476/480664581.py:20: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  trait_codes_df['description'] = trait_codes_df['description'].str.replace('(', ' ')
/tmp/ipykernel_138476/480664581.py:22: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  trait_codes_df['description'] = trait_codes_df['description'].str.replace('[', ' ')
/tmp/ipykernel_1384

#Arthrosis M13_ARTHROSIS
92.29880619049072
#Deforming dorsopathies M13_DEFORMDORSO
77.37968158721924
#Other joint disorders M13_OTHERJOINT
93.77946162223816
#Other necrotizing vasculopathies M13_NECROVASC


KeyboardInterrupt: 

In [3]:
rec_100_snp_pvals_all_traits = []
for i in range(0, 3815, 5):
    pickle_in = open('/home/ctrader/Hold/GWAS/pickled_subthreshold_data/recommended_SNP_pvals_for_each_trait_'+str(i)+'_through_'+str(i+4)+'_CT20220124.pickle', 'rb')
    rec_100_snp_pvals_5_traits=pickle.load(pickle_in)
    rec_100_snp_pvals_all_traits.extend(rec_100_snp_pvals_5_traits)

In [4]:
rec_100_snps_all_traits = []
for i in range(0, 3815, 5):
    pickle_in = open('/home/ctrader/Hold/GWAS/pickled_subthreshold_data/recommended_SNPs_for_each_trait_'+str(i)+'_through_'+str(i+4)+'_CT20220124.pickle', 'rb')
    rec_100_snps_5_traits=pickle.load(pickle_in)
    rec_100_snps_all_traits.extend(rec_100_snps_5_traits)

In [5]:
phen_list = []
for i in range(0, 3815, 5):
    pickle_in = open('/home/ctrader/Hold/GWAS/pickled_subthreshold_data/phenotype_list_for_recs_'+str(i)+'_through_'+str(i+4)+'_CT20220124.pickle', 'rb')
    x=pickle.load(pickle_in)
    phen_list.extend(x)

In [73]:
ts[ts['trait_ind']==1]['Trait'].iloc[0]

'#Deforming dorsopathies'

In [76]:
ts[ts['snp_ind']==1857466]['Reference SNP cluster ID'].iloc[0]

'rs761308380'

In [7]:
t = d[d['Trait']=='Radial styloid tenosynovitis [de Quervain]']
# t[t['Reference SNP cluster ID']=='rs761308380']
t

,Chromosome,p1,p2,Reference SNP cluster ID,p-value,Trait


In [194]:
t = d[d['p-value']=='8.97135e-15']
t

,Chromosome,p1,p2,Reference SNP cluster ID,p-value,Trait
2141,chr1,999705,999706,rs144665333,8.97135e-15,#Deforming dorsopathies
8057520,chr21,33432280,33432281,rs201843430,8.97135e-15,"Diagnoses - main ICD10: J40 Bronchitis, not sp..."


In [208]:
phenotype = 'M13_DEQUERVAIN'
trait_data = []
with gzip.open('/home/nasi4/proj/GWAS/UKBB_201807/phenotypes/' + phenotype + '.gwas.imputed_v3.both_sexes/' + phenotype + '.hg38.bed.gz','r') as f:
    for line in f:
        trait_data.append(line.decode("utf-8").strip('\n').split('\t'))
trait_dfM13_DEQUERVAIN = pd.DataFrame(trait_data, columns=['Chromosome', 'Start', 'End', 'Reference SNP cluster ID', 'p-value'])
trait_dfM13_DEQUERVAIN['p-value'] = trait_dfM13_DEQUERVAIN['p-value'].astype('float')

In [210]:
trait_dfM13_DEQUERVAIN[trait_dfM13_DEQUERVAIN['p-value']==8.97135e-15]

,Chromosome,Start,End,Reference SNP cluster ID,p-value
957,chr1,999705,999706,rs144665333,8.971350e-15


In [204]:
M13_DEFORMDORSOtrait_df[M13_DEFORMDORSOtrait_df['p-value']==8.97135e-15]

,Chromosome,Start,End,Reference SNP cluster ID,p-value


In [209]:
trait_dfM13_DEQUERVAIN[trait_dfM13_DEQUERVAIN['Reference SNP cluster ID']=='rs144665333']

,Chromosome,Start,End,Reference SNP cluster ID,p-value
957,chr1,999705,999706,rs144665333,8.971350e-15


In [207]:
M13_DEFORMDORSOtrait_df[M13_DEFORMDORSOtrait_df['Reference SNP cluster ID']=='rs144665333']

,Chromosome,Start,End,Reference SNP cluster ID,p-value
957,chr1,999705,999706,rs144665333,0.932049


In [203]:
phenotype = 'M13_DEFORMDORSO'
trait_data = []
with gzip.open('/home/nasi4/proj/GWAS/UKBB_201807/phenotypes/' + phenotype + '.gwas.imputed_v3.both_sexes/' + phenotype + '.hg38.bed.gz','r') as f:
    for line in f:
        trait_data.append(line.decode("utf-8").strip('\n').split('\t'))
M13_DEFORMDORSOtrait_df = pd.DataFrame(trait_data, columns=['Chromosome', 'Start', 'End', 'Reference SNP cluster ID', 'p-value'])
M13_DEFORMDORSOtrait_df['p-value'] = M13_DEFORMDORSOtrait_df['p-value'].astype('float')

In [107]:
# rec_100_snps_all_traits[381491]
# phen_list[1]

In [3]:
trait_codes = []
with gzip.open('/home/nasi4/proj/GWAS/UKBB_201807/info/phenotypes.both_sexes.tsv.gz','r') as f:
    for line in f:
        trait_codes.append(line.decode("utf-8").strip('\n').split('\t'))

trait_codes_df = pd.DataFrame(trait_codes[1:], columns=trait_codes[0])

In [4]:
pd.options.display.max_colwidth=1000
trait_codes_df[trait_codes_df['phenotype']=='M13_DEQUERVAIN']

,phenotype,description,variable_type,source,n_non_missing,n_missing,n_controls,n_cases,PHESANT_transformation,notes
1033,M13_DEQUERVAIN,Radial styloid tenosynovitis [de Quervain],categorical,finngen,361194,0,360952,242,NA,NA


In [84]:
rec_100_snp_pvals_all_traits[115]

0.789317

In [51]:
#including p-val information and creating lists for bed file
top_100_recommended_snps_all_traits_chrm_list = []
top_100_recommended_snps_all_traits_start_list = []
top_100_recommended_snps_all_traits_end_list = []
top_100_recommended_snps_all_traits_order_list = []
top_100_recommended_snps_all_traits_id_list = []
top_100_recommended_snps_all_traits_pvals = []
for i in range(0, len(rec_100_snps_all_traits), 100):
    #getting information for all SNP recs for each trait
#     snp_info_df = df_snp_id_based.loc[ts_snp_ind_based.loc[predictions_sorted_100[i]]['Reference SNP cluster ID']]
    per_trait_rec_snps_ids = rec_100_snps_all_traits[i:i+100]
    if per_trait_rec_snps_ids[0] == 'SKIP_CT20220107':
        continue
    snp_info_df = df_snp_id_based.loc[per_trait_rec_snps_ids]
#     if len(snp_info_df) != 100:
#         print(len(snp_info_df))
    top_100_recommended_snps_all_traits_chrm_list.extend(list(snp_info_df['Chromosome']))
    top_100_recommended_snps_all_traits_start_list.extend(list(snp_info_df['p1']))
    top_100_recommended_snps_all_traits_end_list.extend(list(snp_info_df['p2']))
    top_100_recommended_snps_all_traits_order_list.extend(list(i + np.arange(100)))
    top_100_recommended_snps_all_traits_id_list.extend(list(snp_info_df.index))
    top_100_recommended_snps_all_traits_pvals.extend(rec_100_snp_pvals_all_traits[i:i+100])

In [60]:
#double checking results
test = []
for i in rec_100_snps_all_traits:
    if i != 'SKIP_CT20220107':
        test.append(i)
for i, j in zip(test, top_100_recommended_snps_all_traits_id_list):
    if i != j:
        print(i, j)

In [67]:
#double checking results
c=0
for i in range(len(rec_100_snps_all_traits)):
    if rec_100_snps_all_traits[i] != 'SKIP_CT20220107':
        if rec_100_snps_all_traits[i] != top_100_recommended_snps_all_traits_id_list[i-c]:
            print(rec_100_snps_all_traits[i], top_100_recommended_snps_all_traits_id_list[i-c])
        
    else:
        c += 1

In [73]:
#Creating a bed file for top 100 recommended snps for each trait with pvals
today = date.today()
bed_path_name = os.path.join('/home/ctrader/Hold/GWAS/recommendations/', 'GWAS_top_100_recommended_SNPs_for_each_trait_all_traits_with_pvals_CT' + today.strftime("%Y%m%d") + '.bed')
f= open(bed_path_name, 'w')
for i in range(len(top_100_recommended_snps_all_traits_chrm_list)):
    f.write(top_100_recommended_snps_all_traits_chrm_list[i] + '\t' + str(top_100_recommended_snps_all_traits_start_list[i]) + '\t' + str(top_100_recommended_snps_all_traits_end_list[i]) + '\t' + str(top_100_recommended_snps_all_traits_id_list[i]) + '\t' + str(top_100_recommended_snps_all_traits_order_list[i])+ '\t' + str(top_100_recommended_snps_all_traits_pvals[i]) + '\n')
f.close()

**Function to recommend 100 SNPs for each trait**

In [23]:
def generate_100_recs(model):
    #retrieving the trait and SNP biases and embeddings for calculating prediction scores
    snp_b, snp_e = model.get_item_representations()
    snp_e = snp_e.T
    trait_b, trait_e = model.get_user_representations()
    n_dimensions, n_snps = snp_e.shape
    n_traits, n_dimensions = trait_e.shape
    #calculating predictions scores for each trait-SNP interaction
    predictions = ((np.dot(trait_e, snp_e) + snp_b).T + trait_b).T
    #sorting prediction scores to get the ordered SNP recommendations for each trait
    predictions_sorted = np.argsort(-predictions, axis=1)
    #taking the first 100 SNP recommendations for each trait
#     predictions_sorted_100 = predictions_sorted[:, :100]
    predictions_sorted_100_new = np.zeros([n_traits, 100]).astype(int)
    for i in range(len(predictions_sorted)):
        count=0
        for j in predictions_sorted[i]:
            if data_matrix[i, j] == 0:
                predictions_sorted_100_new[i, count]=j
                count+=1
                if count == 100:
                    break
    return predictions_sorted_100

In [46]:
snp_b, snp_e = gwas_model_e30_d10_reg00001.get_item_representations()
snp_e = snp_e.T
trait_b, trait_e = gwas_model_e30_d10_reg00001.get_user_representations()
n_dimensions, n_snps = snp_e.shape
n_traits, n_dimensions = trait_e.shape

predictions = ((np.dot(trait_e, snp_e) + snp_b).T + trait_b).T
predictions_sorted = np.argsort(-predictions, axis=1)

In [49]:
predictions_sorted

array([[1579853, 1136184,  505766, ...,  410638,  102652, 1500625],
       [1397627, 1387002, 1930278, ..., 1008147, 1002570, 2005792],
       [1579853, 1442482, 1579859, ...,  737550,  394384, 1756435],
       ...,
       [ 502746,  513280, 1767920, ...,  673519, 1310764, 1804105],
       [1383034,  262861,  396913, ...,  722791, 1884527, 1864524],
       [1397627, 1387002, 1418775, ...,   55130, 1911456, 1718382]])

In [66]:
predictions_sorted_100_new = np.zeros([n_traits, 100]).astype(int)
for i in range(len(predictions_sorted)):
    count=0
    for j in predictions_sorted[i]:
        if data_matrix[i, j] == 0:
            predictions_sorted_100_new[i, count]=j
            count+=1
            if count == 100:
                break

In [69]:
data_matrix[1, 1857466]

0.04958729701043407

In [68]:
print(predictions_sorted_100_new[1])
print(predictions_sorted_100[1])

[1397627 1387002 1930278 1418775 1807023 1960888 1994614 1974248 1014146
 1427574 1251707  531659  217756 1307683  187615 1957431 1501996  259467
 1479849 1079079  554607  530615 1347770 1343291  899003  174302 1889070
 1279249  901851 1276607  349512 1137141  237365  502746  385965  505011
  892580  536178  276845 1523221 1289027 1515393  891663  224331  239325
 1593960 1144914 1316766  609535 1169629  908562  272923  470157  201808
  227478 1976750 1465526 1017339    1826  441780  453443  200295  266409
 1065875  474180 1992110  642890   80522  794212  262785 1549497 1904039
 1059298 1135334  675304  248547  617976  476645 1774873 1309903 1139639
  563917  432996 1591950 1142883 2000191 1312701   82933  261145  610749
  861081 1534337  266034 1591095  544875 1593687  270405 1037492 1294143
 1819062]
[1397627 1387002 1930278 1418775 1807023 1960888 1994614 1974248 1014146
 1427574 1251707  531659  217756 1307683  187615 1857466 1957431 1501996
  259467 1479849 1079079  554607  530615 

**Function to find the top 100 subthreshold SNPs for each trait**

In [4]:
trait_codes = []
with gzip.open('/home/nasi4/proj/GWAS/UKBB_201807/info/phenotypes.both_sexes.tsv.gz','r') as f:        
    for line in f:
        trait_codes.append(line.decode("utf-8").strip('\n').split('\t'))

trait_codes_df = pd.DataFrame(trait_codes[1:], columns=trait_codes[0])

In [5]:
trait_codes_df['description'] = trait_codes_df['description'].str.replace(')', ' ')
trait_codes_df['description'] = trait_codes_df['description'].str.replace('(', ' ')
trait_codes_df['description'] = trait_codes_df['description'].str.replace('-', ' ')
trait_codes_df['description'] = trait_codes_df['description'].str.replace('[', ' ')
trait_codes_df['description'] = trait_codes_df['description'].str.replace(']', ' ')
trait_codes_df['description'] = trait_codes_df['description'].str.replace('+', ' ')
# trait_codes_df['description'] = trait_codes_df['description'].str.replace('  ', ' ')
trait_codes_df['description'] = trait_codes_df['description'].str.replace('_', ' ')
trait_codes_df['description'] = trait_codes_df['description'].str.replace(' ', '')

# trait_codes_df['description'] = trait_codes_df['description'].str.replace('/', '')

/tmp/ipykernel_139643/3393187603.py:1: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  trait_codes_df['description'] = trait_codes_df['description'].str.replace(')', ' ')
/tmp/ipykernel_139643/3393187603.py:2: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  trait_codes_df['description'] = trait_codes_df['description'].str.replace('(', ' ')
/tmp/ipykernel_139643/3393187603.py:4: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  trait_codes_df['description'] = trait_codes_df['description'].str.replace('[', ' ')
/tmp/ipykernel_1396

In [6]:
phenotype_list = []
not_added = []
for original_trait in trait_values:
    trait = original_trait.replace(')', ' ').replace('(', ' ').replace('-', ' ').replace('[', ' ').replace(']', ' ').replace('+', ' ').replace('_', ' ').replace(' ', '')
    if len(trait_codes_df[trait_codes_df['description']==(trait)])==1:
        phenotype_list.append(list(trait_codes_df[trait_codes_df['description']==(trait)]['phenotype'])[0])
    elif ((len(trait_codes_df[trait_codes_df['description']==(trait)])==2) & (len(trait_codes_df[(trait_codes_df['description']==(trait)) & (trait_codes_df['phenotype'].str.contains('irnt'))])==1)):
        phenotype_list.append(list(trait_codes_df[(trait_codes_df['description']==(trait)) & (trait_codes_df['phenotype'].str.contains('irnt'))]['phenotype'])[0])
    else:
        phenotype_list.append('SKIP_CT20220107')
        not_added.append(original_trait)

In [23]:
print(len(phenotype_list))

3815


In [24]:
top_100_SNPs_for_each_trait = []

In [27]:
phenotype_list[4]

'NECROVASC_FG'

In [35]:
for phenotype, trait in zip(phenotype_list[4:5], trait_values[4:5]):
    if phenotype != 'SKIP_CT20220107':
        t = time.time()
        print(trait, phenotype)
        trait_data = []
        with gzip.open('/home/nasi4/proj/GWAS/UKBB_201807/phenotypes/' + phenotype + '.gwas.imputed_v3.both_sexes/' + phenotype + '.hg38.bed.gz','r') as f:        
            for line in f:        
                trait_data.append(line.decode("utf-8").strip('\n').split('\t'))
        trait_df = pd.DataFrame(trait_data, columns=['Chromosome', 'Start', 'End', 'Reference SNP cluster ID', 'p-value'])
        trait_df['p-value'] = trait_df['p-value'].astype('float')
        trait_df = trait_df[trait_df['p-value']!=0]
        trait_df = trait_df.sort_values(['p-value']).reset_index()
        top_100_SNPs_for_each_trait.append(trait_df.iloc[:100])
        print(time.time()-t)
    else:
        top_100_SNPs_for_each_trait.append('SKIP_CT20220107')

#Other necrotizing vasculopathies (FG) NECROVASC_FG
181.69249606132507


In [40]:
trait_df[trait_df['p-value']!=0].iloc[:100]

,index,Chromosome,Start,End,Reference SNP cluster ID,p-value
9,976661,chr1,236571429,236571430,rs75511846,2.736333e-319
10,5902312,chr2,24308650,24308651,rs6744320,1.209370e-256
11,3508740,chr14,20583944,20583945,rs142227946,6.958250e-248
12,10589441,chr6,35291697,35291698,rs150457131,1.099200e-230
13,5902209,chr2,24275805,24275806,rs77507357,1.371270e-226
...,...,...,...,...,...,...
104,1753877,chr11,4999646,4999647,rs76895431,7.491240e-47
105,3938230,chr15,23645131,23645132,rs74004212,1.046520e-46
106,932653,chr1,228158538,228158539,rs549807766,1.766610e-46
107,7507769,chr22,42140283,42140284,rs183760077,2.191870e-46


In [7]:
top_100_subthreshold_snps_all_traits = []
for i in range(0, 3815, 5):
    pickle_in = open('/home/ctrader/Hold/GWAS/pickled_subthreshold_data/top_100_SNPs_for_each_trait_'+str(i)+'_through_'+str(i+4)+'_CT20220110.pickle', 'rb')
    top_100_snps_5_traits=pickle.load(pickle_in)
    top_100_subthreshold_snps_all_traits.extend(top_100_snps_5_traits)

In [11]:
pickle_out = open('/home/ctrader/Hold/GWAS/pickled_subthreshold_data/top_100_SNPs_for_each_trait_all_traits_CT20220112.pickle', 'wb')
pickle.dump(top_100_subthreshold_snps_all_traits, pickle_out)
pickle_out.close()

In [70]:
top_100_subthreshold_snps_all_traits_chrm_list = []
top_100_subthreshold_snps_all_traits_start_list = []
top_100_subthreshold_snps_all_traits_end_list = []
top_100_subthreshold_snps_all_traits_pvalue_list = []
top_100_subthreshold_snps_all_traits_order_list = []
top_100_subthreshold_snps_all_traits_id_list = []
for i in range(len(top_100_subthreshold_snps_all_traits)):
    if isinstance(top_100_subthreshold_snps_all_traits[i], pd.core.frame.DataFrame):
        top_100_subthreshold_snps_all_traits_chrm_list.extend(list(top_100_subthreshold_snps_all_traits[i]['Chromosome']))
        top_100_subthreshold_snps_all_traits_start_list.extend(list(top_100_subthreshold_snps_all_traits[i]['Start']))
        top_100_subthreshold_snps_all_traits_end_list.extend(list(top_100_subthreshold_snps_all_traits[i]['End']))
        top_100_subthreshold_snps_all_traits_pvalue_list.extend(list(top_100_subthreshold_snps_all_traits[i]['p-value']))
        top_100_subthreshold_snps_all_traits_order_list.extend(list(100*i + np.arange(100)))
        top_100_subthreshold_snps_all_traits_id_list.extend(list(top_100_subthreshold_snps_all_traits[i]['Reference SNP cluster ID']))

In [71]:
#Creating a bed file for top 100 subthreshold snps for each trait
today = date.today()
bed_path_name = os.path.join('/home/ctrader/Hold/GWAS/recommendations/', 'GWAS_top_100_subthreshold_SNPs_for_each_trait_all_traits_CT' + today.strftime("%Y%m%d") + '.bed')
f= open(bed_path_name, 'w')
for i in range(len(top_100_subthreshold_snps_all_traits_chrm_list)):
    f.write(top_100_subthreshold_snps_all_traits_chrm_list[i] + '\t' + str(top_100_subthreshold_snps_all_traits_start_list[i]) + '\t' + str(top_100_subthreshold_snps_all_traits_end_list[i]) + '\t' + str(top_100_subthreshold_snps_all_traits_pvalue_list[i]) + '\t' + str(top_100_subthreshold_snps_all_traits_id_list[i]) + '\t' + str(top_100_subthreshold_snps_all_traits_order_list[i]) + '\n')
f.close()

In [217]:
trait_codes_df[trait_codes_df['description']==('3mm asymmetry angle (left)')]

,phenotype,description,variable_type,source,n_non_missing,n_missing,n_controls,n_cases,PHESANT_transformation,notes
3846,5111_irnt,3mm asymmetry angle (left),continuous_irnt,phesant,73818,287376,NA,NA,5111_0|| INTEGER || CONTINUOUS || IRNT ||,This is the asymmetric angle of keratometry re...
4120,5111_raw,3mm asymmetry angle (left),continuous_raw,phesant,73818,287376,NA,NA,5111_0|| INTEGER || CONTINUOUS || ||,This is the asymmetric angle of keratometry re...


In [224]:
list(trait_codes_df[(trait_codes_df['description']==('3mm asymmetry angle (left)')) & (trait_codes_df['phenotype'].str.contains('irnt'))]['phenotype'])[0]

'5111_irnt'

In [185]:
for trait in trait_values:
    if len(trait_codes_df[trait_codes_df['description']==(trait)])==2:

        if len(trait_codes_df[(trait_codes_df['description']==(trait)) & (trait_codes_df['phenotype'].str.contains('irnt'))])!=1:
            print(trait)

Asthma
Mesothelioma
Snoring


In [231]:
for i, j in zip(trait_values, phenotype_list):
    print(i, j)

#Arthrosis M13_ARTHROSIS
#Deforming dorsopathies M13_DEFORMDORSO
#Other joint disorders M13_OTHERJOINT
#Other necrotizing vasculopathies M13_NECROVASC
#Other necrotizing vasculopathies (FG) NECROVASC_FG
#Polyarthropathies M13_POLYARTHROPATHIES
#Spondylopathies M13_SPONDYLOPATHY
3mm asymmetry angle (left) 5111_irnt
3mm asymmetry angle (right) 5108_irnt
3mm asymmetry index (left) 5156_irnt
3mm asymmetry index (right) 5159_irnt
3mm cylindrical power (left) 5119_irnt
3mm cylindrical power (right) 5116_irnt
3mm cylindrical power angle (left) 5112_irnt
3mm cylindrical power angle (right) 5115_irnt
3mm index of best keratometry results (left) 5292
3mm index of best keratometry results (right) 5237
3mm regularity index (left) 5163_irnt
3mm regularity index (right) 5160_irnt
3mm strong meridian (left) 5135_irnt
3mm strong meridian (right) 5132_irnt
3mm strong meridian angle (left) 5104_irnt
3mm strong meridian angle (right) 5107_irnt
3mm weak meridian (left) 5096_irnt
3mm weak meridian (right) 

Diagnoses - main ICD10: Z45 Adjustment and management of implanted device Z45
Diagnoses - main ICD10: Z46 Fitting and adjustment of other devices Z46
Diagnoses - main ICD10: Z47 Other orthopaedic follow-up care Z47
Diagnoses - main ICD10: Z48 Other surgical follow-up care Z48
Diagnoses - main ICD10: Z50 Care involving use of rehabilitation procedures Z50
Diagnoses - main ICD10: Z51 Other medical care Z51
Diagnoses - main ICD10: Z52 Donors of organs and tissues Z52
Diagnoses - main ICD10: Z53 Persons encountering health services for specifie procedures, not carried out Z53
Diagnoses - main ICD10: Z71 Persons encountering health services for other counselling and medical advice, not elsewhere classified Z71
Diagnoses - main ICD10: Z76 Persons encountering health services in other circumstances Z76
Diagnoses - main ICD10: Z80 Family history of malignant neoplasm Z80
Diagnoses - main ICD10: Z85 Personal history of malignant neoplasm Z85
Diagnoses - main ICD10: Z87 Personal history of other

Malignant neoplasm of small intestine SKIP_CT20220107
Malignant neoplasm of stomach SKIP_CT20220107
Malignant neoplasm of testis SKIP_CT20220107
Malignant neoplasm of thyroid gland SKIP_CT20220107
Malignant neoplasm of tonsil SKIP_CT20220107
Malignant neoplasm of urinary organs C3_URINARY_TRACT
Malignant neoplasm, without specification of site SKIP_CT20220107
Manic/hyper symptoms: All of the above 6156_15
Manic/hyper symptoms: I needed less sleep than usual 6156_13
Manic/hyper symptoms: I was more creative or had more ideas than usual 6156_14
Manifestations of mania or irritability: I needed less sleep than usual 20548_5
Manifestations of mania or irritability: I was easily distracted 20548_7
Manifestations of mania or irritability: I was more active than usual 20548_9
Manifestations of mania or irritability: I was more confident than usual 20548_8
Manifestations of mania or irritability: I was more creative or had more ideas than usual 20548_6
Manifestations of mania or irritability: 

Treatment/medication code: vascalpha 5mg m/r tablet 20003_1141190160
Treatment/medication code: venlafaxine 20003_1140916282
Treatment/medication code: ventolin 100micrograms inhaler 20003_1140861998
Treatment/medication code: ventolin 2mg tablet 20003_1140862060
Treatment/medication code: verapamil 20003_1140888510
Treatment/medication code: vesicare 5mg tablet 20003_1141200384
Treatment/medication code: viscotears liquid eye gel 20003_1140865010
Treatment/medication code: vitamin a 20003_1140852756
Treatment/medication code: vitamin b compound tablet 20003_1140871024
Treatment/medication code: vitamin b1 preparation 20003_1140909872
Treatment/medication code: vitamin b12 preparation 20003_1140870570
Treatment/medication code: vitamin b6 preparation 20003_1140909874
Treatment/medication code: vitamin c product SKIP_CT20220107
Treatment/medication code: vitamin d product 20003_1140870954
Treatment/medication code: vitamin e product 20003_1140871112
Treatment/medication code: voltarol 2